# Chest X-ray inference

Run **Setup** before presenting. Then use the three demonstrations in slide order:
First prediction → What went into the model? → Try another image.

The slides first introduce our pretrained model: TorchXRayVision 1.5.7,
DenseNet-121 with `densenet121-res224-all` weights. Its training combines labeled
chest X-ray datasets, including NIH ChestX-ray14, PadChest and CheXpert.
See `README.md` for model/training references and the exact input
preparation and output interpretation. `xray_model.py` supplies the loader
and `prepare_image()` helper; the README also attributes the demo images.


## Setup — before class

Install the requirements and run `prepare_xray_demo.py` once from this
folder. See `README.md` for the environment and provenance.
The setup cell loads the trained TorchXRayVision model. Run it before the first
Demo slide; the class begins with the image, not this setup code.


In [ ]:
import imageio.v3 as iio
import numpy as np
import pandas as pd
import torch
import torchxrayvision as xrv
from PIL import Image
from IPython.display import display
from xray_model import load_model, prepare_image, sample_paths

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
paths = sample_paths()
path = paths[0]
model = load_model()
selected = ["Cardiomegaly", "Nodule"]


## First prediction

Can this model find signs of an enlarged heart? What else does it score?


In [ ]:
with Image.open(path) as original:
    display(original.copy())


In [ ]:
# The helper prepares this first input. We inspect those steps next.
first_input = prepare_image(path).unsqueeze(0)
with torch.inference_mode():
    first_scores = model(first_input)

first_findings = pd.Series(first_scores[0].tolist(), index=model.pathologies)
display(first_findings.loc[selected].round(3).rename("Model score"))


Cardiomegaly means an enlarged heart; a nodule is a small rounded spot in a lung.
These are two selected finding scores, not the two largest outputs.

Pause here for **The first prediction** in the slides.


## What went into the model?

A grayscale file stores one intensity per pixel. Inspect the same image array.


In [ ]:
pixels = iio.imread(path)
print("Array:", pixels.shape, pixels.dtype)
print("One pixel:", pixels[100, 200])


In [ ]:
scaled = xrv.datasets.normalize(pixels, 255)
channel = scaled[None, ...]
cropped = xrv.datasets.XRayCenterCrop()(channel)
resized = xrv.datasets.XRayResizer(224)(cropped)
image_tensor = torch.from_numpy(resized)

print("One image:", image_tensor.shape, image_tensor.dtype)
print("Prepared intensity range:", image_tensor.min().item(), image_tensor.max().item())


In [ ]:
# Undo the intensity scaling for display. Resizing still reduced detail.
display_pixels = ((resized[0] + 1024) * 255 / 2048).clip(0, 255).astype(np.uint8)
display(Image.fromarray(display_pixels))


## Run the model

Add a batch axis, then run the prepared image as in the first demonstration.


In [ ]:
batch = image_tensor.unsqueeze(0)
model.eval()
with torch.inference_mode():
    scores = model(batch)

print("Input:", batch.shape, "Output:", scores.shape)
findings = dict(zip(model.pathologies, scores[0].tolist()))
print("Cardiomegaly:", round(findings["Cardiomegaly"], 3))
print("Nodule:", round(findings["Nodule"], 3))


## More than one finding

Several findings can occur together. These scores do not have to sum to one;
the model already transforms its outputs, so we do not apply softmax.
The scores express model support, not a patient's chance of disease.


In [ ]:
print("Sum of all 18 scores:", scores[0].sum().item())


## Try another image

Look at the second image before running it. Will the prepared shapes change?


In [ ]:
with Image.open(paths[1]) as original:
    display(original.copy())


In [ ]:
batch = torch.stack([image_tensor, prepare_image(paths[1])])
with torch.inference_mode():
    scores = model(batch)

print("Input:", batch.shape, "Output:", scores.shape)
table = pd.DataFrame(
    scores.tolist(),
    index=[path.name for path in paths],
    columns=model.pathologies,
).T
display(table.loc[selected].round(3))


The cardiomegaly scores differ substantially; the nodule scores are similar.
Both inputs still have the same prepared dimensions.

Instructor note: these examples have no verified targets bundled here, and
independence from model training is unknown. They demonstrate inference,
not a clinical diagnosis or an accuracy test. Image attribution is in
`README.md`.


## Optional: inspect all 18 findings


In [ ]:
display(table.round(3))
